In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# 1. Load the Audited Data
df = pd.read_csv('anomaly_detection_202605211808.csv')

In [ ]:
# 2. Feature Engineering
# Convert shift_hour (timestamp) to just the hour of the day (0-23)
df['shift_hour'] = pd.to_datetime(df['shift_hour']).dt.hour

In [ ]:
# Create a binary target variable: 1 if 'Critical SLA Breach', 0 if 'On Time'
df['is_delayed'] = (df['operational_status'] == 'Critical SLA Breach').astype(int)

# Define Features (X) and Target (y)
# Updated to lowercase to match PostgreSQL export formatting
X = df[['shift_hour', 'courier_vehicle', 'weather_condition', 'est_distance_km']]
y = df['is_delayed']

In [ ]:
# 3. Train/Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# 4. Build the Preprocessing Engine
numeric_features = ['shift_hour', 'est_distance_km']
numeric_transformer = StandardScaler()

categorical_features = ['courier_vehicle', 'weather_condition']
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [ ]:
# 5. Build the Final Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', max_depth=10, random_state=42))
])

In [ ]:
# 6. Train the Model
print("Training the Random Forest model...\n")
pipeline.fit(X_train, y_train)

In [ ]:
# 7. Evaluate the Performance
y_pred = pipeline.predict(X_test)
print("--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['On Time', 'SLA Breach']))

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))